In [2]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import xesmf as xe
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
LATRANGE = CONFIGS['domain']['latrange']   # [5.0, 25.0]
LONRANGE = CONFIGS['domain']['lonrange']   # [60.0, 90.0]
YEARS    = CONFIGS['domain']['years']      # 2000-2020
MONTHS   = CONFIGS['domain']['months']    # [6, 7, 8] = JJA

In [4]:
# Load ERA5 SST from ARCO-ERA5 public GCP zarr store (same source as other ERA5 variables)
store = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
era5  = xr.open_zarr(store, storage_options=dict(token='anon'))

sst = era5['sea_surface_temperature']  # K, hourly, 0.25°×0.25°

# Standardize coordinates to match calculator.py conventions
sst = sst.rename({'latitude':'lat','longitude':'lon'})
sst = sst.assign_coords(lon=((sst.lon + 180.0) % 360.0 - 180.0)).sortby('lon')

# Select JJA months and 2000-2020, add padding for regridding
lat_pad = (LATRANGE[0] - 4, LATRANGE[1] + 4)
lon_pad = (LONRANGE[0] - 4, LONRANGE[1] + 4)
sst = sst.sel(
    lat=slice(lat_pad[0], lat_pad[1]),
    lon=slice(lon_pad[0], lon_pad[1]),
    time=(sst.time.dt.year.isin(YEARS)) & (sst.time.dt.month.isin(MONTHS))
).load()

print(f'SST shape: {sst.shape},  lat: [{float(sst.lat.min()):.1f}, {float(sst.lat.max()):.1f}],  '
      f'lon: [{float(sst.lon.min()):.1f}, {float(sst.lon.max()):.1f}]')
print(f'SST range: [{float(sst.min()):.1f}, {float(sst.max()):.1f}] K')

ValueError: numpy.nanmin raises on a.size==0 and axis=None; So Bottleneck too.

In [ ]:
# Regrid to 1°×1° — same method as calculator.py DataCalculator.regrid()
target_lats = np.arange(LATRANGE[0], LATRANGE[1] + 1.0, 1.0)
target_lons = np.arange(LONRANGE[0], LONRANGE[1] + 1.0, 1.0)
target_grid = xr.Dataset({'lat':(['lat'], target_lats), 'lon':(['lon'], target_lons)})

regridder = xe.Regridder(sst, target_grid, method='bilinear')
sst_1deg  = regridder(sst, keep_attrs=True)

# JJA time mean over 2000-2020 (convert to °C for readability)
sst_mean_C = sst_1deg.mean('time') - 273.15
print(f'SST 1°×1° mean: [{float(sst_mean_C.min()):.1f}, {float(sst_mean_C.max()):.1f}] °C')

In [ ]:
fig, ax = pplt.subplots(nrows=1, ncols=1, proj='cyl', figwidth=5.0)
ax.format(
    coast=True, borders=False,
    latlim=LATRANGE, lonlim=LONRANGE,
    latlines=5, lonlines=5, grid=False,
    title='ERA5 JJA SST (2000–2020 mean)',
)
m = ax.pcolormesh(
    sst_mean_C.lon, sst_mean_C.lat, sst_mean_C,
    cmap='thermal', levels=15
)
fig.colorbar(m, loc='b', label='SST (°C)')
pplt.show()
# fig.save('../figs/fig_sst_mean.jpg')